In [ ]:
# hide
# no-output
from IPython.utils.capture import capture_output
with capture_output():
    %pip install -q plotly anywidget

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
import icm_plotly
from icm_plotly import RED, BLUE, GOLD, IRON, TEAL, STEEL

Drag $f_s$: the red baseband is the signal's own spectrum, and the blue
copies are the ones that sampling adds at every multiple of $f_s$. The
gold dashes mark $\pm f_s / 2$. Lower $f_s$, or widen the band with
$f_{\max}$, until the copies run into the baseband.

In [ ]:
# hide
# autorun
FS0, FMAX0 = 1600.0, 500.0          # starting parameters

u = np.linspace(-1.0, 1.0, 181)     # one copy's shape, in units of f_max
S = (np.exp(-((np.abs(u) - 0.28) ** 2) / 0.018)
     + 0.55 * np.exp(-((np.abs(u) - 0.68) ** 2) / 0.010))
S *= 1.0 - np.abs(u) ** 8           # taper to zero at the band edges
S /= S.max()

KS = list(range(-4, 5))             # the copies we draw, centered at k * f_s
NK = len(KS)

def verdict(fs, fmax):
    return "clear" if fs > 2 * fmax else "touch" if fs == 2 * fmax else "overlap"

VERDICT = {"clear": "<b>copies clear</b>", "touch": "<b>copies touch</b>",
           "overlap": "<b>copies overlap</b>"}
VERDICT_COLOR = {"clear": IRON, "touch": GOLD, "overlap": RED}

def figure():
    fig = go.Figure()
    for k in KS:
        fig.add_scatter(x=k * FS0 + u * FMAX0, y=S, mode="lines",
                        line=dict(color=BLUE, width=1.3))
    fig.add_scatter(x=u * FMAX0, y=S, mode="lines",   # the baseband, on top
                    line=dict(color=RED, width=2.4))
    fig.add_scatter(x=[-FS0 / 2, -FS0 / 2, None, FS0 / 2, FS0 / 2, None],
                    y=[0, 1.12, None, 0, 1.12, None], mode="lines",
                    line=dict(color=GOLD, width=1.6, dash="dash"))
    state = verdict(FS0, FMAX0)
    fig.add_scatter(x=[2440], y=[1.16], mode="text", textposition="middle left",
                    text=[VERDICT[state]],
                    textfont=dict(size=17, color=VERDICT_COLOR[state]))
    fig.update_xaxes(range=[-2500, 2500], title_text="Frequency (Hz)",
                     fixedrange=True)
    fig.update_yaxes(range=[0, 1.26], title_text="Magnitude", fixedrange=True)
    return fig

def controls(fig):
    fs = widgets.FloatSlider(description="Sample rate $f_s$ (Hz)", min=400,
                             max=2400, value=FS0, step=50)
    fmax = widgets.FloatSlider(description=r"Band limit $f_{\max}$ (Hz)", min=100,
                               max=800, value=FMAX0, step=25)
    readout = widgets.HTML()

    # the defaults snapshot the arrays; the page's notebooks share one kernel
    def update(fs, fmax, u=u, KS=KS, NK=NK, verdict=verdict, VERDICT=VERDICT,
               VERDICT_COLOR=VERDICT_COLOR, readout=readout):
        state = verdict(fs, fmax)
        with fig.batch_update():
            for i, k in enumerate(KS):
                fig.data[i].x = k * fs + u * fmax
            fig.data[NK].x = u * fmax
            fig.data[NK + 1].x = [-fs / 2, -fs / 2, None, fs / 2, fs / 2, None]
            fig.data[NK + 2].text = [VERDICT[state]]
            fig.data[NK + 2].textfont.color = VERDICT_COLOR[state]
        relation = {"clear": "&gt;", "touch": "=", "overlap": "&lt;"}[state]
        readout.value = (f"<span style='font-size:0.9em'><i>f</i><sub>s</sub> = "
                         f"{fs:.0f} Hz &nbsp;{relation}&nbsp; 2<i>f</i><sub>max</sub> = "
                         f"{2 * fmax:.0f} Hz</span>")

    widgets.interactive_output(update, {"fs": fs, "fmax": fmax})
    return widgets.VBox([fs, fmax, readout])

icm_plotly.show(figure, controls)